# Unconditional Diffusion Model — Apples & Oranges (Smoke Test)

Minimal DDPM using HuggingFace `diffusers` on 64x64 fruit images.

In [ ]:
#@title 0 — Install & Imports
!pip install -q diffusers accelerate kagglehub

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from diffusers import DDPMScheduler, DDPMPipeline, UNet2DModel
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import shutil
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
#@title 1 — Download & Prepare Dataset
import kagglehub

dataset_path = kagglehub.dataset_download("moltean/fruits")
print(f"Dataset downloaded to: {dataset_path}")

def prepare_fruit_dataset(source_root, dest_root='./fruit_data'):
    """Flatten apple + orange classes into fruit_data/train/{apple,orange}/"""
    os.makedirs(dest_root, exist_ok=True)

    for split_src, split_dst in [('Training', 'train'), ('Test', 'test')]:
        src_base = os.path.join(source_root, split_src)
        if not os.path.exists(src_base):
            for candidate in Path(source_root).rglob(split_src):
                if candidate.is_dir():
                    src_base = str(candidate)
                    break
        if not os.path.exists(src_base):
            print(f"Warning: Could not find {split_src} directory")
            continue

        for class_dir in sorted(os.listdir(src_base)):
            class_lower = class_dir.lower()
            if 'apple' in class_lower and 'pine' not in class_lower:
                target_class = 'apple'
            elif 'orange' in class_lower:
                target_class = 'orange'
            else:
                continue

            src_dir = os.path.join(src_base, class_dir)
            dst_dir = os.path.join(dest_root, split_dst, target_class)
            os.makedirs(dst_dir, exist_ok=True)

            for img_file in os.listdir(src_dir):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    src_path = os.path.join(src_dir, img_file)
                    dst_path = os.path.join(dst_dir, f"{class_dir}_{img_file}")
                    if not os.path.exists(dst_path):
                        shutil.copy2(src_path, dst_path)

    for split in ['train', 'test']:
        split_dir = os.path.join(dest_root, split)
        if os.path.exists(split_dir):
            for cls in sorted(os.listdir(split_dir)):
                cls_dir = os.path.join(split_dir, cls)
                n = len([f for f in os.listdir(cls_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
                print(f"  {split:>5}/{cls:<10}: {n:>5} images")
    return dest_root

data_root = prepare_fruit_dataset(dataset_path)

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # -> [-1, 1]
])

# ImageFolder on the flattened train/ dir (apple + orange)
dataset = datasets.ImageFolder(os.path.join(data_root, "train"), transform=transform)
print(f"\nTotal training images: {len(dataset)}")

dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2, drop_last=True)

In [ ]:
#@title 2 — Preview Samples
batch = next(iter(dataloader))
images = batch[0][:8]

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    img = (images[i].permute(1, 2, 0) * 0.5 + 0.5).clip(0, 1)
    ax.imshow(img)
    ax.axis("off")
plt.suptitle("Training Samples")
plt.tight_layout()
plt.show()

In [ ]:
#@title 3 — Model & Scheduler
model = UNet2DModel(
    sample_size=64,
    in_channels=3,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 256),
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D",
        "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D",
        "AttnUpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
    ),
).to(device)

scheduler = DDPMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model parameters: {n_params:.1f}M")

In [ ]:
#@title 4 — Training Loop
NUM_EPOCHS = 40
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    for batch in dataloader:
        clean_images = batch[0].to(device)
        bs = clean_images.shape[0]

        # Sample noise and timesteps
        noise = torch.randn_like(clean_images)
        timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (bs,), device=device).long()

        # Add noise to images
        noisy_images = scheduler.add_noise(clean_images, noise, timesteps)

        # Predict noise
        noise_pred = model(noisy_images, timesteps).sample
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS}  loss: {avg_loss:.4f}")

In [ ]:
#@title 5 — Loss Curve
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.tight_layout()
plt.show()

In [ ]:
#@title 6 — Generate Samples
pipeline = DDPMPipeline(unet=model, scheduler=scheduler).to(device)

generated = pipeline(batch_size=8, num_inference_steps=1000, output_type="np").images

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(generated[i].clip(0, 1))
    ax.axis("off")
plt.suptitle("Generated Samples")
plt.tight_layout()
plt.show()